# CSV velocity evaluation pipeline: steps 2 to 7

This notebook is for the CSV-only workflow where the estimate and ground-truth files already exist.

It runs these stages:

2. Plot the raw estimate and GT velocity CSV files
3. Clean the estimate and GT CSV files
4. Align cleaned GT velocity to the cleaned estimate timeline
5. Clip the evaluation window, either by liftoff detection or by manual times
6. Compute bias-corrected velocity errors and RMSE
7. Compute the drift-rate metric and generate drift-fit plots

In [ ]:
from pathlib import Path
import subprocess
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "scripts" / "pipeline").is_dir():
    ROOT = next(
        (parent for parent in ROOT.parents if (parent / "scripts" / "pipeline").is_dir()),
        ROOT,
    )
PIPELINE_DIR = ROOT / "scripts" / "pipeline"
FLIGHT_ID = "flight_4"

GT_CSV = ROOT / "data" / "TL-data" / FLIGHT_ID / "data.csv"
EST_CSV = ROOT / "archive" / "legacy_results" / "old_airio_finetuned_results_souvik" / "Finetune_Air-io_Results_CSV_ori_fix" / f"{FLIGHT_ID}_velocity.csv"

OUTPUT_DIR = ROOT / "results" / "csv_eval_outputs" / FLIGHT_ID
RAW_PLOTS_DIR = OUTPUT_DIR / "plots_raw"
DRIFT_PLOTS_DIR = OUTPUT_DIR / "plots_drift_rate"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CLEAN_SCRIPT = PIPELINE_DIR / "3_vel_csv_dataset_cleaner.py"
PLOT_SCRIPT = PIPELINE_DIR / "2_plot_vel_csv_data.py"
ALIGN_SCRIPT = PIPELINE_DIR / "4_align_vel_csv.py"
LIFTOFF_CLIP_SCRIPT = PIPELINE_DIR / "5_liftoff_and_window_clipping.py"
MANUAL_CLIP_SCRIPT = PIPELINE_DIR / "5_manual_relative_window_clipping.py"
RMSE_SCRIPT = PIPELINE_DIR / "6_bias_zeroing_and_rmse.py"
DRIFT_RATE_SCRIPT = PIPELINE_DIR / "7_plot_absolute_velocity_error.py"

EST_CLEAN_CSV = OUTPUT_DIR / f"{EST_CSV.stem}_clean{EST_CSV.suffix}"
GT_CLEAN_CSV = OUTPUT_DIR / f"{GT_CSV.stem}_clean{GT_CSV.suffix}"
GT_ALIGNED_CSV = OUTPUT_DIR / f"{GT_CLEAN_CSV.stem}_aligned{GT_CLEAN_CSV.suffix}"
EST_WINDOW_CSV = OUTPUT_DIR / f"{EST_CLEAN_CSV.stem}_window{EST_CLEAN_CSV.suffix}"
GT_WINDOW_CSV = OUTPUT_DIR / f"{GT_ALIGNED_CSV.stem}_window{GT_ALIGNED_CSV.suffix}"
BIAS_ERRORS_CSV = OUTPUT_DIR / f"{EST_WINDOW_CSV.stem}_bias_errors{EST_WINDOW_CSV.suffix}"
RMSE_SUMMARY_TXT = OUTPUT_DIR / f"{EST_WINDOW_CSV.stem}_rmse_summary.txt"
DRIFT_FIT_PNG = DRIFT_PLOTS_DIR / f"{BIAS_ERRORS_CSV.stem.replace('_bias_errors', '')}_drift_fit.png"

def run_command(command, cwd=None):
    display_command = " ".join(str(part) for part in command)
    print(f"Running: {display_command}")
    if cwd is not None:
        print(f"Working directory: {cwd}")
    result = subprocess.run(command, text=True, capture_output=True, cwd=cwd)
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr, file=sys.stderr)
    result.check_returncode()

for required_path in [GT_CSV, EST_CSV, CLEAN_SCRIPT, PLOT_SCRIPT, ALIGN_SCRIPT, MANUAL_CLIP_SCRIPT, RMSE_SCRIPT, DRIFT_RATE_SCRIPT]:
    if not required_path.exists():
        raise FileNotFoundError(required_path)

print(f"Flight: {FLIGHT_ID}")
print(f"GT CSV: {GT_CSV}")
print(f"EST CSV: {EST_CSV}")
print(f"Output directory: {OUTPUT_DIR}")

## 3. Clean velocity CSV files

This removes out-of-bounds velocity samples, applies the median-based filtering already defined in the script, interpolates gaps, and writes cleaned CSV files into the output directory.

In [ ]:
run_command([
    "python3",
    str(CLEAN_SCRIPT),
    "--est_csv",
    str(EST_CSV),
    "--gt_csv",
    str(GT_CSV),
    "--out_dir",
    str(OUTPUT_DIR),
])

print(f"Clean estimate CSV: {EST_CLEAN_CSV}")
print(f"Clean GT CSV: {GT_CLEAN_CSV}")

## 4. Align GT to the estimate timeline

This compensates GT latency and resamples the cleaned GT file onto the cleaned estimate timestamps.

In [ ]:
GT_LATENCY = 0.025
TOLERANCE = None

command = [
    "python3",
    str(ALIGN_SCRIPT),
    "--est_csv",
    str(EST_CLEAN_CSV),
    "--gt_csv",
    str(GT_CLEAN_CSV),
    "--out",
    str(GT_ALIGNED_CSV),
    "--gt_latency",
    str(GT_LATENCY),
]

if TOLERANCE is not None:
    command.extend(["--tolerance", str(TOLERANCE)])

run_command(command)
print(f"Aligned GT CSV: {GT_ALIGNED_CSV}")

## 5A. Liftoff-based window clipping

Use this if `vel_z` in the aligned GT CSV gives a reliable liftoff signal. Leave it commented out if you prefer manual clipping in the next cell.

In [ ]:
# LIFT_THRESHOLD = 0.3
# MIN_CONSECUTIVE = 5
# WINDOW_START_AFTER_LIFTOFF = 7.0
# WINDOW_END_AFTER_LIFTOFF = 22.0
#
# run_command([
#     "python3",
#     str(LIFTOFF_CLIP_SCRIPT),
#     "--est_csv",
#     str(EST_CLEAN_CSV),
#     "--gt_csv",
#     str(GT_ALIGNED_CSV),
#     "--lift_threshold",
#     str(LIFT_THRESHOLD),
#     "--min_consecutive",
#     str(MIN_CONSECUTIVE),
#     "--window_start",
#     str(WINDOW_START_AFTER_LIFTOFF),
#     "--window_end",
#     str(WINDOW_END_AFTER_LIFTOFF),
# ], cwd=OUTPUT_DIR)

## 5B. Manual window clipping

Use this when you already know the evaluation segment. The times are seconds from the start of the aligned GT CSV.

In [ ]:
WINDOW_START = 36.06
WINDOW_END = 39.06

run_command([
    "python3",
    str(MANUAL_CLIP_SCRIPT),
    "--est_csv",
    str(EST_CLEAN_CSV),
    "--gt_csv",
    str(GT_ALIGNED_CSV),
    "--window_start",
    str(WINDOW_START),
    "--window_end",
    str(WINDOW_END),
    "--output_suffix",
    "_window",
], cwd=OUTPUT_DIR)

print(f"Windowed estimate CSV: {EST_WINDOW_CSV}")
print(f"Windowed GT CSV: {GT_WINDOW_CSV}")

## 6. Bias zeroing and RMSE

Run this after either clipping method in Step 5. It merges the windowed estimate and GT files on time, computes raw and bias-corrected errors, and saves both the per-sample error CSV and the RMSE summary text file.

In [ ]:
print(f"Using estimate window: {EST_WINDOW_CSV}")
print(f"Using GT window: {GT_WINDOW_CSV}")

run_command([
    "python3",
    str(RMSE_SCRIPT),
    "--est_csv",
    str(EST_WINDOW_CSV),
    "--gt_csv",
    str(GT_WINDOW_CSV),
], cwd=OUTPUT_DIR)

print(f"Bias errors CSV: {BIAS_ERRORS_CSV}")
print(f"RMSE summary TXT: {RMSE_SUMMARY_TXT}")

## 7. Drift-rate metric and drift-fit plot

This runs the updated script 7, which computes the horizontal drift-rate metric and generates velocity-error, magnitude, and drift-fit plots from the Step 6 outputs.

In [ ]:
print(f"Using bias-error CSV: {BIAS_ERRORS_CSV}")
print(f"Using estimate window: {EST_WINDOW_CSV}")
print(f"Using GT window: {GT_WINDOW_CSV}")

run_command([
    "python3",
    str(DRIFT_RATE_SCRIPT),
    "--bias_errors_csv",
    str(BIAS_ERRORS_CSV),
    "--est_csv",
    str(EST_WINDOW_CSV),
    "--gt_csv",
    str(GT_WINDOW_CSV),
    "--out_dir",
    str(DRIFT_PLOTS_DIR),
], cwd=OUTPUT_DIR)

print(f"Drift plot directory: {DRIFT_PLOTS_DIR}")
print(f"Expected drift-fit PNG: {DRIFT_FIT_PNG}")

Generated outputs for this notebook are written under `csv_eval_outputs/<flight_id>/`.

Main files after Step 7:

- Raw plots under `plots_raw/`
- Cleaned CSV files for estimate and GT
- Aligned GT CSV
- `*_window.csv` files for the chosen clipping stage
- `*_bias_errors.csv` and `*_rmse_summary.txt`
- Drift-fit plots and the clipped local-Z CSV under `plots_drift_rate/`

## 2. Plot raw CSV data

This creates quick PNG plots from the original estimate and GT CSV files so you can inspect the raw signals before cleaning.

In [ ]:
run_command([
    "python3",
    str(PLOT_SCRIPT),
    "--est_csv",
    str(EST_WINDOW_CSV),
    "--gt_csv",
    str(GT_WINDOW_CSV),
    "--out_dir",
    str(RAW_PLOTS_DIR),
])